In [1]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("/scratch/jczhang/submeso_data_driven").resolve()))

from src.utils import make_grid
from src.utils import build_coarse_grid 
from src.utils import read_mitgcm 
import xarray as xr 
import numpy as np 
import gsw 
import importlib 
from src.utils import coarsen_UVWTS 
from MITgcmutils import mds 
importlib.reload(coarsen_UVWTS) # force reload

def buo_coarse(
    ds_grid: xr.Dataset,
    ds_ocean: xr.Dataset,
    boundary_Y: str = "fill"
) -> xr.DataArray:

    # merge the two dataset
    ds = xr.merge([ds_grid, ds_ocean])
    grid = make_grid.make_grid_periodic_x(ds)
    # water collumn
    # Etabar: sea-surface height (j,i) [m]; broadcast add → z_c(k,j,i)
    z_rel = ds["RC"] - ds["Eta"]  # RC is negative, RC - Eta gives the depth of water column
    
    # TEOS10 EOS
    # model output is salinity and potential temperature
    # gsw function need absolute salinity and conservative temperature
    lat0, lon0 = -55.0, 153.0
    lat2d = xr.full_like(ds["T"].isel(k=0), lat0)               # (j,i)
    lon2d = xr.full_like(ds["T"].isel(k=0), np.mod(lon0, 360.)) # (j,i)

    # TEOS-10 sea pressure from depth (returns dbar)
    p_dbar = xr.apply_ufunc(
        gsw.p_from_z, z_rel, lat2d,
        dask="parallelized",
        output_dtypes=[ds["T"].dtype],
    )


    # --- convert SP, θ -> SA, CT -> ρ ---
    SA  = xr.apply_ufunc(gsw.SA_from_SP, ds["S"], p_dbar, lon2d, lat2d,
                         dask='parallelized', output_dtypes=[ds["S"].dtype])
    CT  = xr.apply_ufunc(gsw.CT_from_pt, SA, ds["T"],
                         dask='parallelized', output_dtypes=[ds["T"].dtype])
    rho = xr.apply_ufunc(gsw.rho, SA, CT, p_dbar,
                         dask='parallelized', output_dtypes=[ds["T"].dtype])  # kg m^-3
    # constants
    g = 9.81 # gravity
    rho0 = 1035 # density
    # buoyancy
    b = (-g * (rho - rho0) / rho0).where(ds["hFacC"] > 0)  # m s^-2
    b.name = "buoyancy"
    b.attrs.update(units="m s^-2", long_name="buoyancy (TEOS-10)")
    
    # ---- buoyancy gradient ----
    # Face wet masks (avoid bleeding across land/blocked faces)
    wetU = ds["hFacW"] > 0
    wetV = ds["hFacS"] > 0
    dbdx_u = (grid.diff(b, "X") / ds["DXC"]).where(wetU)                   # (k,j,i_g)
    dbdy_v = (grid.diff(b, "Y", boundary=boundary_Y) / ds["DYC"]).where(wetV)  # (k,j_g,i)
    return b, dbdx_u, dbdy_v

In [2]:
def _k_name(da): # Find the vertical dim name that matches your data 
    for cand in ("k", "Z", "z", "lev", "k_c"): 
        if cand in da.dims: 
            return cand 
            # fallback: assume first dim is vertical (adjust if needed) 
    return da.dims[0]

def mld_integrals_for_MEI( dbdx_u: xr.DataArray, # (k,j,i_g) ∂b/∂x at U 
                           dbdy_v: xr.DataArray, # (k,j_g,i) ∂b/∂y at V 
                           kpphbl_C: xr.DataArray, # (j,i) ML depth at C 
                           ds_grid: xr.Dataset, 
                           grid, 
                           sigma_is_buoyancy: bool = True, 
                           g: float = 9.81, 
                           rho0: float = 1035.0, 
                         ): 
    # ---- 0) Standardize the vertical index 'k' everywhere ---- 
    RF = ds_grid["RF"] # (k_p1), negative downward , 50
    K = RF.sizes["k_p1"] - 1  # 49
    k_index = xr.DataArray(np.arange(K), dims=("k",), name="k") 
    ku = _k_name(dbdx_u) 
    kv = _k_name(dbdy_v) 
    dbdx_u = (dbdx_u.rename({ku: "k"}) .assign_coords(k=k_index) .reset_coords(drop=True)) 
    dbdy_v = (dbdy_v.rename({kv: "k"}) .assign_coords(k=k_index) .reset_coords(drop=True)) 
    # Mixed-layer depth at faces 
    H_C = kpphbl_C.clip(min=0.0) 
    H_U = grid.interp(H_C, "X", to="left") # (j,i_g) 
    H_V = grid.interp(H_C, "Y", to="left") # (j_g,i) 
    # ---- 1) Interfaces as positive-down depths with SAME 'k' coords ---- 
    ztop1d = (-RF.isel(k_p1=slice(0, -1))).rename(k_p1="k").assign_coords(k=k_index) # 0 to Nr -1
    zbot1d = (-RF.isel(k_p1=slice(1, None))).rename(k_p1="k").assign_coords(k=k_index) # 1 to Nr
    # Broadcast to faces (will use positional matching now that 'k' is unified) 
    ztop_U = ztop1d.broadcast_like(dbdx_u) # (k,j,i_g) 
    zbot_U = zbot1d.broadcast_like(dbdx_u) 
    ztop_V = ztop1d.broadcast_like(dbdy_v) # (k,j_g,i) 
    zbot_V = zbot1d.broadcast_like(dbdy_v) 
    # Partial-cell thickness on faces 
    thkU = (zbot_U - ztop_U) * ds_grid["hFacW"] 
    thkV = (zbot_V - ztop_V) * ds_grid["hFacS"] 
    # ---- 2) Overlap thickness with [0, H_face] (no xr.minimum) ---- 
    H3U = H_U.broadcast_like(dbdx_u) 
    H3V = H_V.broadcast_like(dbdy_v) 
    # min(H, zbot) = where(H <= zbot, H, zbot) 
    minHU = xr.where(H3U <= zbot_U, H3U, zbot_U) 
    minHV = xr.where(H3V <= zbot_V, H3V, zbot_V) 
    wU = xr.where(H3U > ztop_U, minHU - ztop_U, 0.0) 
    wV = xr.where(H3V > ztop_V, minHV - ztop_V, 0.0) 
    # clip and enforce partial thickness 
    wU = xr.where(wU > 0, xr.where(wU < thkU, wU, thkU), 0.0) 
    wV = xr.where(wV > 0, xr.where(wV < thkV, wV, thkV), 0.0) 
    # ---- 3) Integrals over 0..H (∑ sigma * dzLoc) ---- 
    intX = (dbdx_u.fillna(0.0) * wU).sum("k") # (j,i_g) 
    intY = (dbdy_v.fillna(0.0) * wV).sum("k") # (j_g,i) 
    # ---- 4) Convert if sigma are density-gradients (yours are buoyancy-gradients) ---- 
    if sigma_is_buoyancy: 
        dBuoyX_Hu, dBuoyY_Hv = intX, intY 
        out_units = "s^-2 m" 
    else: 
        fac = -g / rho0 
        dBuoyX_Hu, dBuoyY_Hv = fac * intX, fac * intY 
        out_units = "kg m^-4 s^-2" 
    # Optional: mask dry faces 
    wetU = (ds_grid["hFacW"] > 0).any("k") 
    wetV = (ds_grid["hFacS"] > 0).any("k") 
    dBuoyX_Hu = dBuoyX_Hu.where(wetU) 
    dBuoyY_Hv = dBuoyY_Hv.where(wetV) 
    dBuoyX_Hu.name, dBuoyY_Hv.name = "dBuoyX_Hu", "dBuoyY_Hv" 
    for da in (dBuoyX_Hu, dBuoyY_Hv): 
        da.attrs["long_name"] = "ML integral of horizontal buoyancy gradient" 
        da.attrs["units"] = out_units 
    return dBuoyX_Hu, dBuoyY_Hv

In [3]:
import numpy as np 
import xarray as xr 
def compute_Lf_MEI( dBuoyX_Hu: xr.DataArray, # (j, i_g) 
                    dBuoyY_Hv: xr.DataArray, # (j_g, i) 
                    b: xr.DataArray, # (k, j, i) buoyancy at C 
                    H_C: xr.DataArray, # (j, i) ML depth at C 
                    f_eff_C: xr.DataArray, # (j, i) 
                    ds_grid: xr.Dataset, 
                    grid, 
                    g: float = 9.81, 
                    rho0: float = 1035.0, 
                    LfMin: float = 1000.0, ): 
    # --- A) Standardize vertical coord 'k' everywhere ----------------------- 
    RF = ds_grid["RF"] # (k_p1) 
    K = RF.sizes["k_p1"] - 1 
    k_idx = xr.DataArray(np.arange(K), dims=("k",), name="k") 
    def to_k(da): 
        # rename vertical dim to 'k' if needed, then assign common index 
        for cand in ("k", "Z", "z", "lev", "k_c"): 
            if cand in da.dims: 
                if cand != "k": 
                    da = da.rename({cand: "k"}) 
                break 
        if "k" in da.dims: 
                da = da.assign_coords(k=k_idx).reset_coords(drop=True) 
        return da 
    b = to_k(b) 
    hFacC = to_k(ds_grid["hFacC"]) # (k,j,i) with unified k 
    # (dBuoyX_Hu, dBuoyY_Hv, H_C, f_eff_C are 2D—no k to fix) 
    # interfaces as positive-down depths with SAME 'k' coords 
    ztop1d = (-RF.isel(k_p1=slice(0, -1))).rename(k_p1="k").assign_coords(k=k_idx) 
    zbot1d = (-RF.isel(k_p1=slice(1, None))).rename(k_p1="k").assign_coords(k=k_idx) 
    ztop = ztop1d.broadcast_like(b) 
    zbot = zbot1d.broadcast_like(b) 
    # --- 1) MsqH at centers from face integrals averaged to C --------------- 
    dBx_C = grid.interp(dBuoyX_Hu, 'X') # U->C 
    dBy_C = grid.interp(dBuoyY_Hv, 'Y') # V->C 
    MsqH = np.sqrt(0.25*(dBx_C**2 + dBy_C**2)) # s^-2 m 
    # --- 2) NHmixLay at C via ML overlap (honor partial cells) -------------- 
    thkC = (zbot - ztop) * hFacC # partial thickness 
    H3C = H_C.assign_coords().broadcast_like(b) # (k,j,i) with unified k 
    # min(H,zbot) without xr.minimum (avoids version issues) 
    minHC = xr.where(H3C <= zbot, H3C, zbot) 
    wC = xr.where(H3C > ztop, minHC - ztop, 0.0) 
    wC = xr.where(wC > 0, xr.where(wC < thkC, wC, thkC), 0.0) 
    # ∂b/∂z at C using interface spacing (downward-positive dz) 
    db = b - b.shift(k=1) 
    dz = (ztop - ztop.shift(k=1)) 
    sigmaR = (db / dz).where(dz.notnull()) 
    sigmaR_pos = xr.where(sigmaR < 0.0, -sigmaR, 0.0) 
    NH_term = (sigmaR_pos * wC).sum("k") # units: s^-2 * m 
    NHmixLay = np.sqrt(NH_term * H_C.clip(min=0.0)).fillna(0.0) 
    # --- 3) Lf at centers --------------------------------------------------- 
    fC = xr.where(f_eff_C > 0, f_eff_C, np.nan) 
    cand1 = MsqH / (fC**2) # m 
    cand2 = NHmixLay / fC # m 
    Lf_center = xr.ufuncs.maximum( xr.ufuncs.maximum(cand1.fillna(0.0), cand2.fillna(0.0)), LfMin ).rename("Lf") 
    Lf_center.attrs.update(long_name="MEI length scale at centers", units="m") 
    # --- 4) centers -> faces for streamfunction denominators ---------------- 
    Lf_U = grid.interp(Lf_center, 'X', to='left') # C->U 
    Lf_V = grid.interp(Lf_center, 'Y', to='left') # C->V 
    Lf_U.name, Lf_V.name = "Lf_U", "Lf_V" 
    return Lf_center, Lf_U, Lf_V

In [30]:
# modules 
from src.utils import read_mitgcm 
from src.utils import coarsen_UVWTS 
from src.utils import build_coarse_grid 
from src.utils import make_grid
import numpy as np 
import xarray as xr 
from MITgcmutils import mds 
from src.utils import mld_avg

# ---------------------------- parameters -------------------------------------
t = 18458496

box_i = box_j = 10 
# ---------------------------- MITgcm Path ------------------------------------- 
path_scratch = '/scratch/jczhang/' 
path_MITgcm = 'MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run/'
grid_dir = path_scratch + path_MITgcm 
# ---------------------------- grid ------------------------------------- 
ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir) 
ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine,box_j=box_j, box_i=box_i) 
grid = make_grid.make_grid_periodic_x(ds_coarse) 
# ---------------------------- Ce, ds ------------------------------------- 
# efficiency coefficient Ce 
C_e = 1.011
# hard upper cap - useless for my case, since dx < 111 km 
submeso_Lmax = 10000 # m 
# delta_s: local coarse model gridscale dimension 
delta_sx = ds_coarse["DXC"].clip(max=submeso_Lmax) # (j, i_g) 
delta_sy = ds_coarse["DYC"].clip(max=submeso_Lmax) # (j_g, i) 
# ---------------------------- mixed layer depth ------------------------------------ 
# mixed layer depth 
diagMXL = mds.rdmds(path_scratch + path_MITgcm + 'Diags/' +'DiagMXL_2d', t) 
kpphbl = xr.DataArray(diagMXL[1, :,:], dims = ("j","i"), name = "kpphbl") 
kpphbl_coarse = coarsen_UVWTS.coarse_field("kpphbl", kpphbl, ds_fine, box_j, box_i) 
H_U = grid.interp(kpphbl_coarse, 'X', to='left') # (j, i_g) 
H_V = grid.interp(kpphbl_coarse, 'Y', to='left') # (j_g, i) 
# ---------------------------- buoyancy gradient and buoyancy ------------------------------------- 
T = read_mitgcm._ocean_var(grid_dir, "T", t, ("k","j","i")) 
T_coarse = coarsen_UVWTS.coarse_field("T", T, ds_fine, box_j, box_i) 
S = read_mitgcm._ocean_var(grid_dir, "S", t, ("k","j","i")) 
S_coarse = coarsen_UVWTS.coarse_field("S", S, ds_fine, box_j, box_i) 
Eta = read_mitgcm._ocean_var(grid_dir, "Eta", t, ("j","i")) 
Eta_coarse = coarsen_UVWTS.coarse_field("Eta", Eta, ds_fine, box_j, box_i) 
ds_ocean = xr.Dataset(dict(T=T, S=S, Eta=Eta)) 
ds_ocean_c = xr.Dataset(dict(T=T_coarse, S=S_coarse, Eta=Eta_coarse)) 
b, dBdx_u, dBdy_V = buo_coarse(ds_coarse, ds_ocean_c, "fill") 
# integral over mixed layer 
dBuoyX_Hu, dBuoyY_Hv = mld_integrals_for_MEI( dBdx_u, dBdy_V, kpphbl_coarse, ds_coarse, grid, sigma_is_buoyancy=True ) 
# ---------------------------- coriolis and tau------------------------------------- 
f0 = -1.363e-4 # s^-1 
beta = 1.313e-11 # s^-1 m^-1 
subMeso_invTau = 1.6e-6 
YC_da = xr.DataArray(ds_coarse["YC"], dims=('j','i')) 
y_ref = float(YC_da.mean()) 
f = (f0 + beta*(YC_da - y_ref)) # (j,i), s^-1 
f_eff = np.hypot(f, subMeso_invTau) 
f_U = grid.interp(f_eff, 'X', to='left') # (j,i_g) 
f_V = grid.interp(f_eff, 'Y', to='left') # (j_g,i) 
# ---------------------------- Lf = max(M/f, NH/|f|, Lf_min)------------------------------------- 
Lf_center, Lf_U, Lf_V = compute_Lf_MEI(dBuoyX_Hu=dBuoyX_Hu, dBuoyY_Hv=dBuoyY_Hv, b=b, H_C=kpphbl_coarse.clip(min=0.0), f_eff_C=f_eff, ds_grid=ds_coarse, grid=grid, LfMin=3600.0 ) 
# ---------------------------- mu(z)------------------------------------- 
# RF is negative downward (k_p1 interfaces); use it directly. 
rF = ds_coarse["RF"].rename(k_p1="k") # (k) ≤ 0 
rF_U = rF.broadcast_like(dBdx_u) # (k,j,i_g) 
rF_V = rF.broadcast_like(dBdy_V) # (k,j_g,i) 
# Mixed-layer depths on faces (positive) 
H_U = grid.interp(kpphbl_coarse.clip(min=0.0), 'X', to='left') 
H_V = grid.interp(kpphbl_coarse.clip(min=0.0), 'Y', to='left') 
# z2H must be negative inside the ML: [-2, 0] 
z2H_U = xr.where(H_U > 0, 2.0 * rF_U / H_U.broadcast_like(dBdx_u), 0.0) 
z2H_V = xr.where(H_V > 0, 2.0 * rF_V / H_V.broadcast_like(dBdy_V), 0.0) 
def mu_from(z2H): 
    q = (1.0 + z2H)**2 
    mu = (1.0 - q) * (1.0 + (5.0/21.0)*q) 
    return xr.where(mu > 0.0, mu, 0.0) 
muU = mu_from(z2H_U) # (k,j,i_g) 
muV = mu_from(z2H_V) # (k,j_g,i) 
# --- gridscale caps at faces (MIN(dx, Lmax), MIN(dy, Lmax)) --- 
dx_cap_U = ds_coarse["DXC"].clip(max=submeso_Lmax) # already (j, i_g) 
dy_cap_V = ds_coarse["DYC"].clip(max=submeso_Lmax) # already (j_g, i) 
# --- 2D streamfunction amplitudes at faces (depth-integrated) --- 
# bring the "other" horizontal component to the target face
dBuoyY_on_U = grid.interp(dBuoyY_Hv, 'Y')          # j_g,i -> j,i
dBuoyY_on_U = grid.interp(dBuoyY_on_U, 'X', to='left')

dBuoyX_on_V = grid.interp(dBuoyX_Hu, 'X')          # j,i_g -> j,i
dBuoyX_on_V = grid.interp(dBuoyX_on_V, 'Y', to='left')

PsiU0 = ( C_e * dBuoyY_on_U * H_U * dx_cap_U / (Lf_U * f_U) ).fillna(0.0)
PsiV0 = (-C_e * dBuoyX_on_V * H_V * dy_cap_V / (Lf_V * f_V) ).fillna(0.0)

# --- 3D streamfunctions using mu(z/H) --- 
muU_ = muU.transpose('k','j','i_g') # ensure consistent order for broadcasting 
muV_ = muV.transpose('k','j_g','i') 
PsiX = PsiU0.broadcast_like(muU_) * muU_ # (k,j,i_g) 
PsiY = PsiV0.broadcast_like(muV_) * muV_ # (k,j_g,i) 
PsiX.name, PsiY.name = "GM_PsiX_MEI", "GM_PsiY_MEI"

# --- grads of mean T on C, then to W (k,j,i) ---
wetU = ds_coarse["hFacW"] > 0
wetV = ds_coarse["hFacS"] > 0

dTdx_U = (grid.diff(T_coarse, 'X') / ds_coarse["DXC"]).where(wetU)      # (k,j,i_g)
dTdy_V = (grid.diff(T_coarse, 'Y', boundary='fill') / ds_coarse["DYC"]).where(wetV)  # (k,j_g,i)

# bring face-grads to centers
dTdx_C = grid.interp(dTdx_U.fillna(0.0), 'X')    # (k,j,i)
dTdy_C = grid.interp(dTdy_V.fillna(0.0), 'Y')    # (k,j,i)

# average C→W so vertical index matches Ψ(k,·,·) at W levels
dTdx_W = 0.5*(dTdx_C + dTdx_C.shift(k=1))
dTdy_W = 0.5*(dTdy_C + dTdy_C.shift(k=1))

# --- bring Ψ components to centers (same W levels) ---
PsiX_C = grid.interp(PsiX, 'X')    # (k,j,i)
PsiY_C = grid.interp(PsiY, 'Y')    # (k,j,i)

# --- vertical eddy temp flux from cross product ----------------------------
Fz_T_cross = (PsiX_C * dTdy_W - PsiY_C * dTdx_W).rename("Fz_T_cross")   # (k,j,i)
Fz_T_cross.attrs.update(long_name="(Ψ × ∇T)_z", units="K m s^-1?  see note")

# (optional) convert to heat units
rho0, cp = 1035.0, 3990.0
Qz_T_cross = (rho0 * cp) * Fz_T_cross

Qz_T_cross.name = "Qz_T_cross"
Qz_T_cross.attrs.update(units="W m^-2")
submeso_wT_FK = mld_avg.mld_avg(Qz_T_cross, kpphbl_coarse, ds_coarse)
# 
dbdx_C = grid.interp(dBdx_u.fillna(0.0), 'X')
dbdy_C = grid.interp(dBdy_V.fillna(0.0), 'Y')

dbdx_W = 0.5 * (dbdx_C + dbdx_C.shift(k=1))
dbdy_W = 0.5 * (dbdy_C + dbdy_C.shift(k=1))

Fz_b = (PsiX_C * dbdy_W - PsiY_C * dbdx_W).rename("Fz_b")
Fz_b_ml = mld_avg.mld_avg(Fz_b, kpphbl_coarse, ds_coarse)

# Plot

In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.utils import coarsen_UVWTS
from src.utils import read_mitgcm
from src.utils import build_coarse_grid
import numpy as np
from matplotlib import pyplot as plt
import cmocean.cm as cmo
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from src.utils import wT
import xarray as xr
from MITgcmutils import mds
from src.utils import mld_avg
import importlib
importlib.reload(wT)   # force reload
importlib.reload(read_mitgcm) 
importlib.reload(build_coarse_grid) 
#  MITgcm path
# ---------------------------- MITgcm Path ------------------------------------- 
path_scratch = '/scratch/jczhang/' 
path_MITgcm = 'MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run/'
grid_dir = path_scratch + path_MITgcm
# time index
nStep1, nStep2, dS = 18458496, 18458496, 1
time_index = np.arange(nStep1, nStep2 + 1, dS, dtype=int)
num_data = time_index.size
print("Total data number is:", num_data)
# choose a vertical level
k = 0
box_i, box_j=10, 10
# get the grid variables and create the ds
ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine,box_j=box_j, box_i=box_i)

# coordinate variable
XC = ds_fine["XC"].values
YC = ds_fine["YC"].values
XCc = ds_coarse["XC"].values
YCc = ds_coarse["YC"].values

# limits function
def sym_limits(Z, q = 0.995):
    v = np.nanquantile(np.abs(Z), q)
    return float(v)

def pos_limits(Z, q=0.995):
    v = np.nanquantile(Z, q)
    return float(v)

W = read_mitgcm._ocean_var(grid_dir, "W", time_index[-1], ("k_l","j","i"))
W_coarse = coarsen_UVWTS.coarse_field("W", W, ds_fine, box_j, box_i)
T = read_mitgcm._ocean_var(grid_dir, "T", time_index[-1], ("k","j","i"))
T_coarse = coarsen_UVWTS.coarse_field("T", T, ds_fine, box_j, box_i)

ds_ocean = xr.Dataset(dict(W=W, T=T))
ds_ocean_c = xr.Dataset(dict(W=W_coarse, T=T_coarse))

_, submeso_wT_coarse = wT.submeso_vertical_heat_flux(ds_fine, ds_coarse, ds_ocean, ds_ocean_c, box_j = 10, box_i = 10,boundary_Z = "fill", to_wm2 = True)

#  mixed layer avergae
diagMXL = mds.rdmds(path_scratch + path_MITgcm + 'Diags/' +'DiagMXL_2d', time_index[-1])
kpphbl = xr.DataArray(diagMXL[1, :,:], dims = ("j","i"), name = "kpphbl")
kpphbl_coarse = coarsen_UVWTS.coarse_field("kpphbl", kpphbl, ds_fine, box_j, box_i)
submeso_wT_coarse_mld = mld_avg.mld_avg(submeso_wT_coarse, kpphbl_coarse, ds_coarse)

# div_coarse_mld[2403 + _ ,:,:]
fields = {
    # "submesoscale wT":  {"Z": strain_fine[k,:,:],    "units": r"m s$^{-1}$",    "kind": "signed", "cmap": cmo.balance, "X": XC/1e3, "Y": YC/1e3 },
    "submeso_wT coarse":  {"Z": submeso_wT_coarse[k,40:,:],    "units": r"W m^-2$",    "kind": "signed", "cmap": cmo.balance, "X": XCc[40:,:]/1e3, "Y": YCc[40:,:]/1e3},
    "submeso_wT coarse mld":  {"Z": submeso_wT_coarse_mld[40:,:],  "units": r"W m^-2$", "kind": "signed", "cmap": cmo.balance, "X": XCc[40:,:]/1e3, "Y": YCc[40:,:]/1e3},
    "submeso_wT coarse mld Fox Kemper":  {"Z": submeso_wT_FK[40:,:],  "units": r"W m^-2$", "kind": "signed", "cmap": cmo.balance, "X": XCc[40:,:]/1e3, "Y": YCc[40:,:]/1e3},
}

# Compute norms/limits per field
for key, meta in fields.items():    # key is the variable name, meta is the dict contains the variables, units...
    Z = meta["Z"]
    if meta["kind"] == "signed":
        q = sym_limits(Z)
        meta["norm"] = TwoSlopeNorm(vcenter=0.0, vmin=-q, vmax=+q)
    elif meta["kind"] == "positive":
        q = pos_limits(Z)
        meta["norm"] = Normalize(vmin=0.0, vmax=q)
    elif meta["kind"] == "seq":
        meta["norm"] = Normalize(vmin=np.nanmin(Z), vmax=np.nanmax(Z))
    else:
        meta["norm"] = None

# Plot: 2×5
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True, squeeze=False)
axes = axes.ravel()

for ax, (title, meta) in zip(axes, fields.items()):
    X = meta["X"]; Y = meta["Y"]; Z = meta["Z"]; 
    cmap = meta["cmap"]
    norm = meta["norm"]
    pc = ax.pcolormesh(X, Y, Z, shading='auto', cmap=cmap, norm=norm)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(title, fontsize=11, pad=4)
    ax.set_xlabel('x km'); ax.set_ylabel('y km')
    cb = fig.colorbar(pc, ax=ax, pad=0.02)
    cb.ax.set_title(meta["units"], pad=4, fontsize=9)

fig.suptitle(f"submesoscale heat flux (t={time_index}) — fine vs coarse", fontsize=13)


Total data number is: 1


KeyboardInterrupt: 

In [23]:
import numpy as np

def r2_image(y_true, y_pred, eps: float = 1e-12) -> float:
    yt = np.asarray(y_true, dtype=np.float64).ravel()
    yp = np.asarray(y_pred, dtype=np.float64).ravel()

    if yt.shape != yp.shape:
        raise ValueError(f"Shape mismatch: {np.shape(y_true)} vs {np.shape(y_pred)}")

    # remove NaN / inf pairs
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[mask]
    yp = yp[mask]

    if yt.size == 0:
        return float("nan")

    yt_mean = yt.mean()
    ss_tot = np.sum((yt - yt_mean) ** 2)

    if ss_tot <= eps:
        return float("nan")

    ss_res = np.sum((yt - yp) ** 2)
    return float(1.0 - ss_res / ss_tot)




r2 = r2_image(submeso_wT_coarse_mld[62:118, 2:58], submeso_wT_FK[62:118, 2:58])
print("R2 =", r2)

R2 = 0.1797269519561664


In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.utils import coarsen_UVWTS
from src.utils import read_mitgcm
from src.utils import build_coarse_grid
import numpy as np
from matplotlib import pyplot as plt
import cmocean.cm as cmo
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import Normalize
from src.utils import wT
import xarray as xr
from MITgcmutils import mds
from src.utils import mld_avg
import importlib
importlib.reload(wT)   # force reload
importlib.reload(read_mitgcm) 
importlib.reload(build_coarse_grid) 
#  MITgcm path
# ---------------------------- MITgcm Path ------------------------------------- 
path_scratch = '/scratch/jczhang/' 
path_MITgcm = 'MITgcm_baroclinic_instability_old/MITgcm/verification/tutorial_reentrant_channel/run/'
grid_dir = path_scratch + path_MITgcm
# time index
nStep1, nStep2, dS = 18458496, 18458496, 1
time_index = np.arange(nStep1, nStep2 + 1, dS, dtype=int)
num_data = time_index.size
print("Total data number is:", num_data)
# choose a vertical level
k = 0
box_i, box_j=10, 10
# get the grid variables and create the ds
ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine,box_j=box_j, box_i=box_i)

# coordinate variable
XC = ds_fine["XC"].values
YC = ds_fine["YC"].values
XCc = ds_coarse["XC"].values
YCc = ds_coarse["YC"].values

# limits function
def sym_limits(Z, q = 0.995):
    v = np.nanquantile(np.abs(Z), q)
    return float(v)

def pos_limits(Z, q=0.995):
    v = np.nanquantile(Z, q)
    return float(v)

W = read_mitgcm._ocean_var(grid_dir, "W", time_index[-1], ("k_l","j","i"))
W_coarse = coarsen_UVWTS.coarse_field("W", W, ds_fine, box_j, box_i)
T = read_mitgcm._ocean_var(grid_dir, "T", time_index[-1], ("k","j","i"))
T_coarse = coarsen_UVWTS.coarse_field("T", T, ds_fine, box_j, box_i)

ds_ocean = xr.Dataset(dict(W=W, T=T))
ds_ocean_c = xr.Dataset(dict(W=W_coarse, T=T_coarse))

_, submeso_wT_coarse = wT.submeso_vertical_heat_flux(ds_fine, ds_coarse, ds_ocean, ds_ocean_c, box_j = 10, box_i = 10,boundary_Z = "fill", to_wm2 = True)

#  mixed layer avergae
diagMXL = mds.rdmds(path_scratch + path_MITgcm + 'Diags/' +'DiagMXL_2d', time_index[-1])
kpphbl = xr.DataArray(diagMXL[1, :,:], dims = ("j","i"), name = "kpphbl")
kpphbl_coarse = coarsen_UVWTS.coarse_field("kpphbl", kpphbl, ds_fine, box_j, box_i)
submeso_wT_coarse_mld = mld_avg.mld_avg(submeso_wT_coarse, kpphbl_coarse, ds_coarse)

Total data number is: 1


In [17]:
import numpy as np
import xarray as xr
from MITgcmutils import mds

from src.utils import read_mitgcm
from src.utils import coarsen_UVWTS
from src.utils import build_coarse_grid
from src.utils import make_grid
from src.utils import mld_avg

# You already have these in your notebook/module:
# buo_coarse, mld_integrals_for_MEI, compute_Lf_MEI

def r2_image(y_true, y_pred, eps: float = 1e-12) -> float:
    yt = np.asarray(y_true, dtype=np.float64).ravel()
    yp = np.asarray(y_pred, dtype=np.float64).ravel()

    if yt.shape != yp.shape:
        raise ValueError(f"Shape mismatch: {np.shape(y_true)} vs {np.shape(y_pred)}")

    m = np.isfinite(yt) & np.isfinite(yp)
    yt = yt[m]; yp = yp[m]
    if yt.size == 0:
        return float("nan")

    ss_tot = np.sum((yt - yt.mean()) ** 2)
    if ss_tot <= eps:
        return float("nan")

    ss_res = np.sum((yt - yp) ** 2)
    return float(1.0 - ss_res / ss_tot)


def compute_fk_mld_heat_flux(
    *,
    t: int,
    box_i: int,
    box_j: int,
    grid_dir: str,
    C_e: float,
    submeso_Lmax: float = 111000.0,
    subMeso_invTau: float = 1.6e-6,
    f0: float = -1.363e-4,
    beta: float = 1.313e-11,
    LfMin: float = 1000.0,
):
    """
    Returns 2D mixed-layer-averaged FK vertical heat flux (W/m^2) for a given C_e.
    """
    # grid
    ds_fine = read_mitgcm.read_mitgrid_staggered(grid_dir)
    ds_coarse = build_coarse_grid.build_coarse_grid_staggered(ds_fine, box_j=box_j, box_i=box_i)
    grid = make_grid.make_grid_periodic_x(ds_coarse)

    # MLD
    diagMXL = mds.rdmds(grid_dir + "Diags/" + "DiagMXL_2d", t)
    kpphbl = xr.DataArray(diagMXL[1, :, :], dims=("j", "i"), name="kpphbl")
    kpphbl_coarse = coarsen_UVWTS.coarse_field("kpphbl", kpphbl, ds_fine, box_j, box_i)

    H_U = grid.interp(kpphbl_coarse.clip(min=0.0), "X", to="left")
    H_V = grid.interp(kpphbl_coarse.clip(min=0.0), "Y", to="left")

    # ocean vars
    T = read_mitgcm._ocean_var(grid_dir, "T", t, ("k", "j", "i"))
    S = read_mitgcm._ocean_var(grid_dir, "S", t, ("k", "j", "i"))
    Eta = read_mitgcm._ocean_var(grid_dir, "Eta", t, ("j", "i"))
    T_coarse = coarsen_UVWTS.coarse_field("T", T, ds_fine, box_j, box_i)
    S_coarse = coarsen_UVWTS.coarse_field("S", S, ds_fine, box_j, box_i)
    Eta_coarse = coarsen_UVWTS.coarse_field("Eta", Eta, ds_fine, box_j, box_i)
    ds_ocean_c = xr.Dataset(dict(T=T_coarse, S=S_coarse, Eta=Eta_coarse))

    # buoyancy + gradients
    b, dBdx_u, dBdy_V = buo_coarse(ds_coarse, ds_ocean_c, "fill")

    # ML integrals
    dBuoyX_Hu, dBuoyY_Hv = mld_integrals_for_MEI(
        dBdx_u, dBdy_V, kpphbl_coarse, ds_coarse, grid, sigma_is_buoyancy=True
    )

    # coriolis + tau limiter
    YC_da = xr.DataArray(ds_coarse["YC"], dims=("j", "i"))
    y_ref = float(YC_da.mean())
    f = (f0 + beta * (YC_da - y_ref))
    f_eff = np.hypot(f, subMeso_invTau)
    f_U = grid.interp(f_eff, "X", to="left")
    f_V = grid.interp(f_eff, "Y", to="left")

    # Lf
    Lf_center, Lf_U, Lf_V = compute_Lf_MEI(
        dBuoyX_Hu=dBuoyX_Hu,
        dBuoyY_Hv=dBuoyY_Hv,
        b=b,
        H_C=kpphbl_coarse.clip(min=0.0),
        f_eff_C=f_eff,
        ds_grid=ds_coarse,
        grid=grid,
        LfMin=LfMin,
    )

    # mu(z/H)
    rF = ds_coarse["RF"].rename(k_p1="k")
    rF_U = rF.broadcast_like(dBdx_u)
    rF_V = rF.broadcast_like(dBdy_V)

    z2H_U = xr.where(H_U > 0, 2.0 * rF_U / H_U.broadcast_like(dBdx_u), 0.0)
    z2H_V = xr.where(H_V > 0, 2.0 * rF_V / H_V.broadcast_like(dBdy_V), 0.0)

    def mu_from(z2H):
        q = (1.0 + z2H) ** 2
        mu = (1.0 - q) * (1.0 + (5.0 / 21.0) * q)
        return xr.where(mu > 0.0, mu, 0.0)

    muU = mu_from(z2H_U)
    muV = mu_from(z2H_V)

    # caps
    dx_cap_U = ds_coarse["DXC"].clip(max=submeso_Lmax)
    dy_cap_V = ds_coarse["DYC"].clip(max=submeso_Lmax)

    # cross-gradient mapping to faces
    dBuoyY_on_U = grid.interp(dBuoyY_Hv, "Y")
    dBuoyY_on_U = grid.interp(dBuoyY_on_U, "X", to="left")

    dBuoyX_on_V = grid.interp(dBuoyX_Hu, "X")
    dBuoyX_on_V = grid.interp(dBuoyX_on_V, "Y", to="left")

    PsiU0 = (C_e * dBuoyY_on_U * H_U * dx_cap_U / (Lf_U * f_U)).fillna(0.0)
    PsiV0 = (-C_e * dBuoyX_on_V * H_V * dy_cap_V / (Lf_V * f_V)).fillna(0.0)

    # 3D Psi
    PsiX = PsiU0.broadcast_like(muU) * muU
    PsiY = PsiV0.broadcast_like(muV) * muV

    # T gradients
    wetU = ds_coarse["hFacW"] > 0
    wetV = ds_coarse["hFacS"] > 0
    dTdx_U = (grid.diff(T_coarse, "X") / ds_coarse["DXC"]).where(wetU)
    dTdy_V = (grid.diff(T_coarse, "Y", boundary="fill") / ds_coarse["DYC"]).where(wetV)
    dTdx_C = grid.interp(dTdx_U.fillna(0.0), "X")
    dTdy_C = grid.interp(dTdy_V.fillna(0.0), "Y")
    dTdx_W = 0.5 * (dTdx_C + dTdx_C.shift(k=1))
    dTdy_W = 0.5 * (dTdy_C + dTdy_C.shift(k=1))

    PsiX_C = grid.interp(PsiX, "X")
    PsiY_C = grid.interp(PsiY, "Y")

    Fz_T = PsiX_C * dTdy_W - PsiY_C * dTdx_W  # K m/s

    rho0, cp = 1035.0, 3990.0
    Qz_T = rho0 * cp * Fz_T  # W/m^2

    return mld_avg.mld_avg(Qz_T, kpphbl_coarse, ds_coarse)


def tune_four_params(
    *,
    submeso_wT_true_2d,
    region=(slice(62, 118), slice(2, 58)),
    Ce_vals=None,
    invTau_vals=None,
    LfMin_vals=None,
    Lmax_vals=None,
    **fk_kwargs,
):
    """
    Grid search over (Ce, invTau, LfMin, Lmax). Returns a list of dict rows with R2 and params.
    """
    js, is_ = region
    y_true = np.asarray(submeso_wT_true_2d)[js, is_]

    rows = []
    for LfMin in LfMin_vals:

        # compute base once at Ce=1.0 (or any fixed Ce) and scale by Ce
        Ce0 = 1.0
        fk_base = compute_fk_mld_heat_flux(
            C_e=Ce0,
            subMeso_invTau=invTau_vals,
            LfMin=float(LfMin),
            submeso_Lmax=Lmax_vals,
            **fk_kwargs,
        )
        y_base = np.asarray(fk_base)[js, is_]

        for Ce in Ce_vals:
            y_pred = float(Ce) * y_base  # since Ce0=1.0
            r2 = r2_image(y_true, y_pred)
            rows.append(
                {
                    "R2": float(r2),
                    "Ce": float(Ce),
                    "invTau": invTau_vals,
                    "LfMin": float(LfMin),
                    "Lmax": Lmax_vals,
                }
            )

    # sort best first
    rows.sort(key=lambda d: np.nan_to_num(d["R2"], nan=-1e99), reverse=True)
    return rows


# ---------------------------- run tuning -------------------------------------
# You must provide:
#   submeso_wT_coarse_mld  (2D truth field, e.g. ML-avg coarse "truth")
#   t, box_i, box_j, grid_dir

Ce_vals = np.linspace(0.3, 3.0, 20)              # 6 evenly
invTau_vals = 1.16e-6    # 6 evenly (this is 1/tau)
LfMin_vals = np.linspace(3600, 3600.0, 1)      # 6 evenly
Lmax_vals = 10000

rows = tune_four_params(
    submeso_wT_true_2d=submeso_wT_coarse_mld,
    region=(slice(62, 118), slice(2, 58)),
    Ce_vals=Ce_vals,
    invTau_vals=invTau_vals,
    LfMin_vals=LfMin_vals,
    Lmax_vals=Lmax_vals,
    t=18458496,
    box_i=box_i,
    box_j=box_j,
    grid_dir=grid_dir,
)

# Print top 30 as a table
print("\nTop 30 parameter sets by R2:")
print(f"{'rank':>4}  {'R2':>10}  {'Ce':>6}  {'invTau(s^-1)':>12}  {'LfMin(m)':>10}  {'Lmax(m)':>10}")
for k, row in enumerate(rows[:30], start=1):
    print(f"{k:4d}  {row['R2']:10.6f}  {row['Ce']:6.3f}  {row['invTau']:12.3e}  {row['LfMin']:10.1f}  {row['Lmax']:10.1f}")

best = rows[0]
print(
    f"\nBEST: R2={best['R2']:.6f} | Ce={best['Ce']:.4f} | invTau={best['invTau']:.3e} | "
    f"LfMin={best['LfMin']:.1f} | Lmax={best['Lmax']:.1f}"
)


Top 30 parameter sets by R2:
rank          R2      Ce  invTau(s^-1)    LfMin(m)     Lmax(m)
   1    0.251265   1.011     1.160e-06      3600.0     10000.0
   2    0.246069   0.868     1.160e-06      3600.0     10000.0
   3    0.243542   1.153     1.160e-06      3600.0     10000.0
   4    0.227954   0.726     1.160e-06      3600.0     10000.0
   5    0.222899   1.295     1.160e-06      3600.0     10000.0
   6    0.196919   0.584     1.160e-06      3600.0     10000.0
   7    0.189337   1.437     1.160e-06      3600.0     10000.0
   8    0.152966   0.442     1.160e-06      3600.0     10000.0
   9    0.142856   1.579     1.160e-06      3600.0     10000.0
  10    0.096092   0.300     1.160e-06      3600.0     10000.0
  11    0.083455   1.721     1.160e-06      3600.0     10000.0
  12    0.011135   1.863     1.160e-06      3600.0     10000.0
  13   -0.074104   2.005     1.160e-06      3600.0     10000.0
  14   -0.172263   2.147     1.160e-06      3600.0     10000.0
  15   -0.283340   2.289 

In [8]:
import numpy as np

def r2_global(y_true, y_pred):
    """
    Global R² over all samples and pixels.
    y_true, y_pred: np.ndarray with same shape (N, H, W) or (N, 1, H, W)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # remove singleton channel if present
    if y_true.ndim == 4 and y_true.shape[1] == 1:
        y_true = y_true[:, 0, ...]
        y_pred = y_pred[:, 0, ...]
    
    # mask NaNs (e.g. land)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true_flat = y_true[mask]
    y_pred_flat = y_pred[mask]

    # standard R² = 1 - SSR / SST
    ss_res = np.sum((y_true_flat - y_pred_flat)**2)
    ss_tot = np.sum((y_true_flat - y_true_flat.mean())**2)

    return 1.0 - ss_res / ss_tot


def r2_per_image(y_true, y_pred):
    """
    R² for each image separately, plus their mean.
    Returns: r2_list (N,), r2_mean (scalar)
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.ndim == 4 and y_true.shape[1] == 1:
        y_true = y_true[:, 0, ...]
        y_pred = y_pred[:, 0, ...]

    N = y_true.shape[0]
    r2_list = np.empty(N, dtype=float)

    for n in range(N):
        yt = y_true[n]
        yp = y_pred[n]

        mask = np.isfinite(yt) & np.isfinite(yp)
        yt_flat = yt[mask]
        yp_flat = yp[mask]

        ss_res = np.sum((yt_flat - yp_flat)**2)
        ss_tot = np.sum((yt_flat - yt_flat.mean())**2)

        r2_list[n] = 1.0 - ss_res / ss_tot

    return r2_list, float(r2_list.mean())

In [25]:
data = np.load("/scratch/jczhang/data_5km/var_6000/submeso_wT.npz")
y_true = data["data"][0:9,]
print(y_true.shape)

(9, 116, 58)


In [18]:
def _fit_z_norm_y(y_train):
    """
    y_train: (N, 1, H, W) or (N, H, W) or (N,)
    Returns scalar mu, sd (broadcastable).
    """
    EPS = 1e-12
    mu = y_train.mean()
    sd = y_train.std(ddof=1) + EPS
    return np.float32(mu), np.float32(sd)

def _apply_z_norm_y(y, mu, sd):
    return (y - mu) / sd
    
y_mu, y_sd = _fit_z_norm_y(y_true)
y_true = _apply_z_norm_y(y_true, y_mu, y_sd)

data_arr_mu, data_arr_sd = _fit_z_norm_y(data_arr)
data_arr = _apply_z_norm_y(data_arr, data_arr_mu, data_arr_sd)

In [13]:
R2_fk_global = r2_global(y_true, data_arr)

-0.9462417244687131


In [33]:
print(R2_fk_global)

-0.01614192723942498
